# bandwidth_monitor_node.py

In [ ]:
CHANGE LOG: bandwidth_monitor_node UPGRADE

---

1. MEASUREMENT APPROACH CHANGE

---

OLD:

* Uses speedtest library (active measurement)
* Performs real internet speed test (heavy operation)

NEW:

* Uses psutil (passive measurement)
* Reads actual network interface counters (lightweight)

IMPACT:

* Much faster (no external test)
* No bandwidth consumption
* Suitable for real-time systems

- Does NOT measure actual internet throughput, only interface usage 


---

2. LATENCY + JITTER ADDED

---

OLD:

* No latency measurement
* No jitter tracking

NEW:

* RTT measured using socket connection to edge_host
(RTT = Round Trip Time 👉 Time taken for a packet to go client → server → back to client)

* Jitter calculated using sliding window (deque)

IMPACT:

* Adds network quality awareness
* Enables intelligent decision-making (e.g., offloading, failover)
* More realistic network monitoring


---

3. DATA RICHNESS (MESSAGE STRUCTURE)

---


OLD:

* Publishes: [upload, download]

NEW:

* Publishes:
  [upload, download, rtt, jitter, network_ok]

IMPACT:

* 2 → 5 parameters
* Adds reliability signal (network_ok)
* Enables downstream intelligence


---

4. SAMPLING STRATEGY

---

OLD:

* Fixed interval (10 seconds)
* Blocking speedtest

NEW:

* Configurable sample_period
* High-frequency sampling using rospy.Rate
(rospy.Rate is used in ROS .Python (rospy) to control loop execution frequency.
👉 It makes your loop run at a fixed rate (Hz).)

* Non-blocking lightweight loop

IMPACT:

* Real-time capability
* Adjustable performance vs accuracy tradeoff

---

5. THREADING MODEL

---

OLD:

* Uses separate thread for measurement

NEW:

* Single loop (no threading)
* ROS-native rate control

IMPACT:

* Simpler design
* No race conditions

---

6. ERROR HANDLING

---

OLD:

* Basic try-catch around speedtest

NEW:

* Handles:

  * Missing interface
  * Socket failures
  * RTT None cases

IMPACT:

* More robust in real environments


---

7. NETWORK HEALTH FLAG

---

OLD:

* No health indicator

NEW:

* network_ok = 1.0 / 0.0 based on RTT availability

IMPACT:

* Simple but powerful signal
* Can trigger fallback logic



---

8. LOGGING IMPROVEMENT

---

OLD:

* Logs every cycle

NEW:

* Uses rospy.loginfo_throttle (every 2 sec)

IMPACT:

* Reduces log spam
* Cleaner debugging


---

9. CONFIGURABILITY

---

OLD:

* Hardcoded behavior

NEW:

* ROS params:

  * interface
  * edge_host
  * sample_period
  * rtt_window

IMPACT:

* Highly flexible
* Easy deployment tuning



---

10. PERFORMANCE & RESOURCE USAGE

---

OLD:

* Heavy CPU + network usage (speedtest)
* Blocks execution

NEW:

* Lightweight (psutil + socket)
* Minimal CPU + network overhead

IMPACT:

* Suitable for low-power systems
* Works with limited GPU/CPU setups


---



## SUMMARY

In [ ]:
1. **Imports**

   * `collections` → used for deque (fixed-size sliding window for RTT).
   * `socket` → used to measure latency via TCP connection.
   * `time` → used for timestamps and interval calculation.
   * `psutil` → fetches network interface stats (bytes sent/received).
   * `rospy` → ROS Python client library.
   * `Float32MultiArray` → message type for publishing network metrics.


2. **Class Initialization (`__init__`)**

   * Initializes ROS node: `"bandwidth_monitor_node"`.
   * Reads parameters:

     * `interface` → network interface (default: wlan0).
     * `edge_host` → target IP for latency check.
     * `sample_period` → time between samples.
     * `rtt_window` → number of RTT samples stored.
   * Creates publisher on topic `/network/bandwidth`.
   * Initializes:

     * `last_counters` → previous network stats.
     * `last_sample_time` → previous timestamp.
     * `rtt_samples` → deque storing RTT values (fixed size).
   * Starts main loop by calling `measure_bandwidth()`.



3. **Bandwidth Sampling (`_sample_bandwidth`)**

   * Fetch current network counters using `psutil`.
   * If interface not found → return (0.0, 0.0).
   * If first run:

     * Store counters + time.
     * Return (0.0, 0.0) (no previous data to compare).
   * Compute time difference `dt`.
   * Calculate:

     * Upload Mbps = (bytes_sent_diff * 8) / (dt * 1e6)
     * Download Mbps = (bytes_recv_diff * 8) / (dt * 1e6)
   * Update stored counters and time.
   * Return non-negative upload & download.



4. **Latency & Jitter Sampling (`_sample_latency`)**

   * Try TCP connection to `(edge_host, port 80)` with 0.2s timeout.
   * Measure RTT:

     * RTT = (connection_time_end - start_time) * 1000 ms.
   * If connection fails → RTT = None.
   * If RTT valid → append to deque.
   * Compute jitter:

     * If ≥2 samples:

       * Take absolute difference between consecutive RTT values.
       * Average them → jitter.
     * Else → jitter = 0.0.
   * Return `(rtt_ms, jitter_ms)`.

5. **Main Loop (`measure_bandwidth`)**

   * Creates loop rate using `rospy.Rate`:

     * Frequency = 1 / sample_period (minimum 1 Hz).
   * Runs until ROS shutdown:

     * Calls:

       * `_sample_bandwidth()`
       * `_sample_latency()`
     * Determines network status:

       * `network_ok = 1.0` if RTT exists else `0.0`.
     * Prepares message:

       * [upload, download, rtt, jitter, network_ok]
       * RTT = -1.0 if unavailable.
       * Values rounded to 2 decimal places.
     * Publishes message to ROS topic.
     * Logs info (throttled every 2 seconds):

       * Upload, download, RTT, jitter, network status.
     * Handles exceptions → logs warning.
     * Sleeps to maintain loop rate.

6. **Program Entry**

   * If file is run directly:

     * Creates instance of `BandwidthMonitorNode`.
     * This automatically starts monitoring loop.

7. **Overall Functionality**

   * Continuously monitors:

     * Upload speed (Mbps)
     * Download speed (Mbps)
     * Latency (RTT in ms)
     * Jitter (variation in RTT)
     * Network availability flag
   * Publishes all metrics as a ROS message.
   * Works in passive mode (no heavy traffic generation).


# tested_load.py

In [ ]:
CHANGE LOG: LOAD BALANCER NODE UPGRADE

---

1. DECISION ENGINE (CORE UPGRADE)

---

OLD:

* Rule-based (if-else)
* Simple thresholds (CPU, bandwidth)

NEW:

* Multi-factor scoring system:

  * edge_score
  * cloud_score
* Uses:

  * latency (RTT, jitter)
  * compute load (CPU/GPU/RAM)
  * scene profile (motion, complexity, change)
  * tracker uncertainty
  * power consumption
  * queue depth

IMPACT:

* Much smarter decisions
* Context-aware routing
* Handles dynamic environments



---

2. NETWORK AWARENESS (BIG CHANGE)

---

OLD:

* Only upload/download bandwidth

NEW:

* Adds:

  * RTT (latency)
  * jitter
  * network_ok flag

IMPACT:

* Real network quality understanding
* Avoids bad offloading decisions

VERDICT: CRITICAL IMPROVEMENT

---

3. SCENE UNDERSTANDING (NEW FEATURE)

---

OLD:

* No scene awareness

NEW:

* profile_frame():

  * motion_score (optical flow)
  * scene_complexity (edges + texture)
  * change_score (frame difference)

IMPACT:

* Detects dynamic vs static scenes
* Improves decision accuracy

VERDICT: HIGH VALUE ADDITION

---

4. DUAL PATH EXECUTION (NEW)

---

OLD:

* Either onboard OR offboard

NEW:

* Supports:

  * onboard
  * offboard
  * lower_resolution
  * dual_path (both)

IMPACT:

* Reliability ↑
* Accuracy ↑
* Latency safety ↑

VERDICT: ADVANCED SYSTEM DESIGN

---

5. CLOUD PIPELINE (MAJOR CHANGE)

---

OLD:

* Direct blocking HTTP request

NEW:

* Queue-based async system:

  * cloud_queue
  * cloud_worker thread
  * inflight tracking

IMPACT:

* Non-blocking execution
* Better throughput
* Avoids frame drops

VERDICT: STRONG BACKEND IMPROVEMENT

---

6. STATE MANAGEMENT (NEW)

---

OLD:

* No structured state

NEW:

* Maintains:

  * edge_state
  * cloud_state
  * fused_state

IMPACT:

* Enables result fusion
* Enables fallback logic
* Tracks freshness

VERDICT: CRITICAL FOR REAL SYSTEMS

---

7. RESULT FUSION (NEW)

---

OLD:

* Uses only one result

NEW:

* Combines:

  * onboard + cloud outputs
  * handles delays
  * fallback when cloud is slow

IMPACT:

* Robust perception
* Handles network delays

VERDICT: PRODUCTION-LEVEL FEATURE

---

8. RESOURCE MONITORING (UPGRADED)

---

OLD:

* tegrastats parsed (basic)

NEW:

* Adds:

  * power averaging (deque)
  * better structured cache
  * thread-safe access

IMPACT:

* Stable metrics
* Power-aware decisions

VERDICT: IMPORTANT IMPROVEMENT

---

9. IMAGE PIPELINE (OPTIMIZED)

---

OLD:

* Fixed resize (320x240 early)

NEW:

* Dynamic resizing:

  * edge_target_width
  * cloud_target_width
  * adaptive scaling

- JPEG quality adapts to bandwidth

IMPACT:

* Better bandwidth utilization
* Flexible quality control

VERDICT: SMART OPTIMIZATION

---

10. ROI-BASED CLOUD SEND (NEW)

---

OLD:

* Full image always sent

NEW:

* ROI extraction + headers

IMPACT:

* Reduces data transfer
* Faster cloud inference

VERDICT: EFFICIENCY BOOST

---

11. LOGGING SYSTEM (UPGRADED)

---

OLD:

* Basic CSV

NEW:

* Logs:

  * RTT, jitter
  * scene metrics
  * scores (edge/cloud)
  * latency
  * decision path

IMPACT:

* Full observability
* Great for analysis & research

VERDICT: BIG UPGRADE

---

12. TRACKING + UNCERTAINTY (NEW)

---

OLD:

* No tracking intelligence

NEW:

* Adds:

  * object velocity estimation
  * uncertainty modeling
  * freshness requirement

IMPACT:

* Smarter decision making
* Enables prediction-ready system

VERDICT: ADVANCED FEATURE

---

13. CACHING SYSTEM (NEW)

---

OLD:

* No reuse of results

NEW:

* publish_cached_result()

IMPACT:

* Saves computation
* Reduces latency

VERDICT: GOOD OPTIMIZATION

---

14. FAULT TOLERANCE (IMPROVED)

---

OLD:

* Basic fallback

NEW:

* Handles:

  * network failure
  * edge failure
  * delayed responses
  * queue overflow

IMPACT:

* System stability ↑

VERDICT: PRODUCTION READY

---

15. CODE COMPLEXITY (TRADE-OFF)

---

OLD:

* Simple (~400 lines)
* Easy to debug

NEW:

* Very complex (~1500+ lines)
* Many interdependent modules

IMPACT:

* Powerful but harder to maintain

- Debugging becomes harder

VERDICT: NECESSARY COMPLEXITY

---

## FINAL SUMMARY

TOTAL MAJOR CHANGES: 15

SYSTEM EVOLUTION:

OLD:
→ Rule-based dispatcher

NEW:
→ Intelligent adaptive distributed system

---

## PERFORMANCE IMPACT

Latency handling:        MUCH BETTER
Decision quality:        MUCH BETTER
Resource efficiency:     BETTER
Scalability:             MUCH BETTER
Robustness:              MUCH BETTER

---

## TRADE-OFFS

* Increased code complexity
* Higher CPU usage (scene profiling)
* More tuning required (thresholds)

---

## FINAL VERDICT

This is NOT an upgrade.

This is a FULL SYSTEM TRANSFORMATION.

You moved from:
→ "if-else routing script"

to:
→ "adaptive edge-cloud orchestration engine"

---

## END


## SUMMARy

In [ ]:
1. **Imports**

   * Core libs → threading, queue, subprocess, signal, time, json, csv, etc.
   * CV/ML → `cv2`, `numpy`
   * Networking → `requests`
   * ROS → `rospy`, message types
   * Purpose → supports vision processing + networking + ROS communication.

2. **APPLICATION_TABLE**

   * Maps each application → latency sensitivity + accuracy priority.
   * Used for decision-making (edge vs cloud).

3. **Helper Functions**

   * `get_application_table(app)` → returns config for app.
   * `clamp(value)` → restricts value between 0 and 1.




4. **Class Initialization (`LoadBalancerNode`)**

   * Initializes ROS node.
   * Sets:

     * frame tracking variables
     * network stats (upload, RTT, jitter)
     * system resource cache (CPU, GPU, RAM, power)
   * Subscribes to:

     * `/network/bandwidth` (from  first file) 
   * Starts background thread → `_poll_tegrastats()`.




5. **Publishers**

   * Control:

     * robot movement (`Twist`)
     * servo control
   * Image topics:

     * lane, yolo, binary, result images
   * Output topics:

     * lane detection
     * object detection
     * traffic detection
   * Logging:

     * start time




6. **Edge/Cloud Config**

   * Reads edge server URL and extracts host/port.
   * Defines:

     * resolution scaling
     * bandwidth thresholds
     * latency thresholds
     * queue limits
     * power limits




7. **State Storage**

   * Maintains:

     * `edge_state` → onboard results
     * `cloud_state` → cloud results
     * `fused_state` → merged results
   * Separate tracking for:

     * lane detection
     * object detection
     * traffic detection




8. **Cloud Queue System**

   * Uses queue for async cloud requests.
   * Starts worker thread `_cloud_worker`.




9. **CSV Logging**

   * Creates log file.
   * Logs:

     * frame id, latency, cpu, gpu, bandwidth, decision, etc.




10. **ROS Subscribers**

* Lane detection result
* Object detection
* Traffic detection
* Camera image stream




11. **System Initialization**

* Waits for:

  * YOLO service
  * servo initialization
* Starts ROS spin loop.




---




12. **Resource Monitoring (`_poll_tegrastats`)**

* Runs `tegrastats` command.
* Parses:

  * CPU usage
  * GPU usage
  * RAM usage
  * power consumption
* Stores rolling average power.




13. **Bandwidth Callback**

* Receives:

  * upload/download
  * RTT
  * jitter
* Updates internal network state.




---

14. **Image Callback**

* Converts ROS image → numpy array.
* Converts BGR → RGB if needed.
* Calls `handle_frame()`.




---

15. **Frame Handling (`handle_frame`)**

* Controls processing frequency.
* Assigns frame ID + timestamp.
* Stores timing for latency measurement.

For each application:

* Gets:

  * system usage
  * scene profile
  * edge availability
* Calls decision engine.
* Based on decision:

  * send to onboard
  * send to edge/cloud
  * or both (dual path)
* Logs decision.




---

16. **Scene Profiling (`profile_frame`)**

* Converts frame → grayscale.
* Computes:

  * motion (optical flow)
  * edge density
  * texture variance
  * scene change vs previous frame
* Outputs:

  * motion_score
  * scene_complexity
  * change_score




---

17. **Decision Engine (`decide_processing_location`)**

* Inputs:

  * latency need
  * accuracy need
  * CPU/GPU/RAM
  * bandwidth
  * RTT + jitter
  * scene complexity
  * power usage

* Computes:

  * edge_score → favors low latency + low load
  * cloud_score → favors high accuracy + bandwidth

* Logic:

  * if network down → onboard
  * if both scores high → dual path
  * else → pick higher score
  * low bandwidth → reduce resolution
  * latency critical → bias to edge

* Output:

  * route (onboard / offboard / dual / low-res)
  * scores
  * cache decision




---

18. **Edge Server Check**

* Sends HTTP HEAD request.
* Marks server available/unavailable.




---

19. **Onboard Processing (`forward_to_onboard`)**

* Resizes frame.
* Converts to ROS image.
* Publishes to correct topic based on app.




---

20. **Cloud Processing (`forward_to_edge`)**

* Prepares payload:

  * resize + JPEG encode
  * attach metadata headers
* Pushes request into queue.




---

21. **Cloud Worker (`_cloud_worker`)**

* Continuously:

  * takes request from queue
  * sends HTTP POST to edge server
  * processes response
  * logs result
* Handles failures → fallback.




---

22. **Cloud Response Processing**

* Parses JSON response.
* Extracts:

  * lane detection
  * object detection
  * traffic detection
* Sends to respective processors.




---

23. **Lane Data Processing**

* Parses lane structure.
* Computes cloud delay.
* Fusion logic:

  * if delay high → use edge result
  * else merge edge + cloud
* Publishes fused result.




---

24. **Detection Processing**

* Converts detections to standard format.
* Calls fusion logic.



---

25. **Fusion Logic (`fuse_detections`)**

* Matches edge & cloud detections:

  * using IoU + distance
* Uses:

  * velocity prediction (project boxes)
* Combines:

  * best match OR fallback
* Outputs fused detections.




---

26. **Tracking + Uncertainty**

* Tracks object velocity.
* Computes uncertainty:

  * based on changes + detection count difference.



---

27. **Cached Results**

* If scene stable → reuse previous results.
* Reduces compute + latency.




---

28. **Fallback System**

* If cloud fails:

  * use onboard results.




---

29. **ROI Optimization**

* Sends only important region to cloud.
* Based on detected objects.




---

30. **Utility Functions**

* IoU calculation
* box projection (motion prediction)
* center distance
* detection normalization




---

31. **Shutdown**

* Stops threads.
* Closes CSV + network session.




---

32. **Main Execution**

* Starts LoadBalancerNode.
* Entire system runs continuously.




---

33. **Overall Functionality**

* Real-time intelligent load balancer:

  * Decides per frame:
    → onboard vs edge vs cloud
  * Uses:

    * network conditions
    * system load
    * scene complexity
  * Optimizes:

    * latency
    * accuracy
    * power
* Supports:

  * fusion (edge + cloud)
  * caching
  * fallback


# lb_input.py

In [ ]:
---

1. NETWORK MEASUREMENT APPROACH

---

OLD:

* Uses speedtest-cli (active measurement)
* Uses ping for RTT

NEW:

* PassiveNetworkProfiler:

  * psutil for bandwidth
  * socket for RTT
  * jitter calculation
  * sliding window

IMPACT:

* No external calls (fast)
* No bandwidth waste




---

2. JITTER INTRODUCTION (NEW)

---

OLD:

* Only RTT

NEW:

* Adds jitter (RTT variance)

IMPACT:

* Better network stability understanding
* Helps decision engine




---

3. NETWORK HEALTH FLAG (NEW)

---

OLD:

* No network status

NEW:

* network_ok (True/False)

IMPACT:

* Simple decision signal
* Enables fallback logic




---

4. RESOURCE MONITORING

---

OLD:

* get_usage_percent():

  * CPU, GPU, RAM

NEW:

* Adds:

  * power_mw
  * structured output

IMPACT:

* Adds power awareness
* Useful for edge devices




---

6. POWER TRACKING (NEW CLASS)

---

OLD:

* No power tracking

NEW:

* RollingPowerTracker:

  * moving average using deque

IMPACT:

* Smooth power readings
* Better decision inputs




---

7. SCENE PROFILING (MAJOR NEW FEATURE)

---

OLD:

* No scene awareness

NEW:

* SceneProfiler:

  * motion (optical flow)
  * scene complexity (edges + texture)
  * change detection

IMPACT:

* Enables intelligent routing
* Adds perception context





---

8. CODE MODULARITY

---

OLD:

* Flat functions

NEW:

* Modular classes:

  * PassiveNetworkProfiler
  * RollingPowerTracker
  (how we track the power avg 👉 we are not measuring power directly — 
    we are reading it from the system (Jetson) via tegrastats)

  * SceneProfiler

IMPACT:

* Cleaner architecture
* Reusable components




---

9. THREAD SAFETY

---

OLD:

* No locks

NEW:

* Uses threading.Lock in network profiler
(👉 It is a mutex (mutual exclusion lock) Used to prevent multiple threads from modifying shared data at the same time.)

IMPACT:

* Safe concurrent access




---

10. BANDWIDTH SAMPLING METHOD

---

OLD:

* External CLI (slow, blocking)

NEW:

* Delta-based calculation using counters

IMPACT:

* Lightweight
* High-frequency capable





---

11. RTT SAMPLING METHOD

---

OLD:

* ping subprocess

NEW:

* socket connection timing

IMPACT:

* Faster
* More controllable timeout




---

12. DEFAULT PROFILER INSTANCE

---

OLD:

* No reuse

NEW:

* _DEFAULT_NETWORK_PROFILER

IMPACT:

* Avoids reinitialization overhead




---

13. UTILITY FUNCTIONS REWORK

---

OLD:

* get_network_bandwidth() → speedtest
* get_rtt() → ping

NEW:

* Uses internal profiler

IMPACT:

* Consistent system behavior
* Faster calls





---

14. MAIN LOOP IMPROVEMENT

---

OLD:

* Basic print system

NEW:

* Shows:

  * power average
  * jitter + RTT
  * structured network output

IMPACT:

* Better debugging



---

## PERFORMANCE IMPACT

Speed:              MUCH FASTER
Resource usage:     MUCH LOWER
Usability:          MUCH BETTER

---


## SUMAARY

In [ ]:
1. **Imports**

   * System → `subprocess`, `threading`, `time`, `socket`
   * Parsing → `re`, `collections`
   * CV → `cv2`, `numpy`
   * System stats → `psutil`
   * Purpose → system monitoring + network profiling + scene analysis.

---

2. **APPLICATION_TABLE**

   * List of applications with:

     * latency sensitivity
     * accuracy priority
   * Used for decision-making (same concept as load balancer).

---

3. **get_application_table(application_name)**

   * Iterates through table.
   * Returns:

     * latency_sensitivity
     * accuracy_priority
   * Returns `None` if not found.

---

4. **System Resource Monitoring (`get_usage_percent`)**

   * Runs `tegrastats` command.
   * Parses output using regex:

     * RAM usage %
     * CPU usage (average across cores)
     * GPU usage %
     * Power consumption (mW)
   * Returns dictionary:

     * cpu_percent
     * gpu_percent
     * ram_percent
     * power_mw
   * On failure → returns all `None`.

---

5. **PassiveNetworkProfiler Class**

   * Purpose → measure network without active load.

---

6. **Initialization**

   * Inputs:

     * network interface (default wlan0)
     * host IP
     * sample window size
   * Initializes:

     * last counters (for bandwidth diff)
     * last timestamp
     * deque for RTT samples (for jitter)

---

7. **Bandwidth Sampling (`sample_bandwidth`)**

   * Uses `psutil.net_io_counters`.
   * If first run → return 0 values.
   * Calculates:

     * upload Mbps
     * download Mbps
   * Formula:

     * (bytes difference * 8) / time
   * Ensures non-negative values.

---

8. **RTT Sampling (`sample_rtt`)**

   * Opens TCP connection to host.
   * Measures:

     * RTT in ms
   * If fails → RTT = None
   * Stores RTT samples in deque.
   * Computes jitter:

     * average absolute difference between consecutive RTT values.
   * Returns:

     * rtt_ms
     * jitter_ms
     * network_ok flag

---

9. **Combined Network Sampling (`sample`)**

   * Calls:

     * sample_bandwidth()
     * sample_rtt()
   * Returns unified network metrics.

---

10. **RollingPowerTracker Class**

* Maintains sliding window of power values.

---

11. **update(power_mw)**

* Adds new power sample.
* Returns updated average.

---

12. **average()**

* Returns mean of stored power values.
* If empty → returns 0.

---

13. **SceneProfiler Class**

* Extracts visual complexity from frames.

---

14. **Initialization**

* Sets:

  * flow resolution (for motion)
  * embedding resolution (for change detection)
* Stores previous frame info.

---

15. **profile(frame_rgb)**

* Converts frame → grayscale.
* Downsamples:

  * for optical flow
  * for embedding

Computes:

a) **Motion Score**
- Uses optical flow (Farneback).
- Measures average motion magnitude.

b) **Scene Complexity**
- Edge density (Canny edges)
- Texture (Laplacian variance)
- Combined into normalized score.

c) **Change Score**
- Difference between current & previous embedding.

* Updates previous frame state.
* Returns:

  * motion_score
  * scene_complexity
  * change_score

---

16. **Global Network Profiler**

* `_DEFAULT_NETWORK_PROFILER`
* Shared instance for quick calls.

---

17. **get_network_bandwidth()**

* Updates interface.
* Returns:

  * upload Mbps
  * download Mbps

---

18. **get_rtt()**

* Updates host.
* Returns:

  * RTT only

---

19. **Main Execution Block**

* Creates:

  * network profiler
  * power tracker

Loop:

* Fetch system usage
* Update power average
* Print:

  * CPU, RAM, GPU
  * power
  * network stats
* Sleeps 2 seconds

---

20. **Overall Functionality**

* This file is a **utility monitoring module**:

  * system resource tracking (CPU, GPU, RAM, power)
  * network profiling (bandwidth, RTT, jitter)
  * scene complexity analysis (motion + texture)
* Designed to support:

  * load balancing decisions
  * performance optimization
  * real-time adaptive systems

---

21. **Key Difference vs LoadBalancer**

* This file:
  → only **collects metrics**
* LoadBalancer:
  → **uses these metrics to decide execution**


# Load_balancer_main.launch

In [ ]:
CHANGE LOG: load_balancer_main.launch UPGRADE

---

1. CODE CLEANUP (FORMATTING)

---

OLD:

* Inline tags (hard to read)
* No spacing
* No structure

NEW:

* Clean formatting
* Proper indentation
* Separated tags

IMPACT:

* Better readability
* Easier maintenance

VERDICT: GOOD PRACTICE (minor but important)

---

2. YOLO NODE PARAMETER ADDITIONS

---

OLD:

* Basic params:

  * engine
  * od_engine
  * conf_thresh

NEW:

* Added:

  * frame_skip = 2
  * cache_ttl = 0.25

IMPACT:

* frame_skip → reduces compute load
* cache_ttl → avoids redundant inference

VERDICT: SMART PERFORMANCE OPTIMIZATION

---

3. BANDWIDTH NODE CONFIGURABILITY

---

OLD:

* No parameters
* Hardcoded behavior

NEW:

* Added params:

  * interface
  * edge_host
  * sample_period
  * rtt_window

IMPACT:

* Fully configurable
* Matches your upgraded monitoring system

VERDICT: IMPORTANT INTEGRATION FIX

---

4. BANDWIDTH THRESHOLD FIX (CRITICAL)

---

OLD:

* bandwidth_high_threshold = 7
* bandwidth_low_threshold = 7  ❌ WRONG

NEW:

* bandwidth_high_threshold = 7
* bandwidth_low_threshold = 2  ✅ FIXED

IMPACT:

* Proper decision boundary
* Avoids incorrect routing

VERDICT: CRITICAL BUG FIX

---

5. EDGE vs CLOUD IMAGE CONFIG (NEW)

---

OLD:

* No resolution control

NEW:

* Added:

  * edge_target_width = 640
  * cloud_target_width = 960
  * cloud_target_height = 1080
  * low_res_scale = 0.65

IMPACT:

* Adaptive image scaling
* Better bandwidth control

VERDICT: MAJOR SYSTEM IMPROVEMENT

---

6. CLOUD COMMUNICATION CONTROL (NEW)

---

OLD:

* No control over latency/queue

NEW:

* Added:

  * edge_timeout = 2.0
  * cloud_delay_threshold = 0.75
  * max_cloud_queue = 4

IMPACT:

* Prevents overload
* Handles slow cloud safely

VERDICT: PRODUCTION-LEVEL FEATURE

---

7. DUAL PATH CONFIG (NEW)

---

OLD:

* No dual execution support

NEW:

* Added:

  * dual_path_threshold = 0.60
  * dual_path_margin = 0.15

IMPACT:

* Enables hybrid execution
* Improves reliability

VERDICT: ADVANCED FEATURE ENABLEMENT

---

8. POWER-AWARE CONTROL (NEW)

---

OLD:

* No power awareness

NEW:

* Added:

  * power_budget_mw = 3200

IMPACT:

* Energy-aware decisions
* Critical for embedded systems

VERDICT: HIGH VALUE ADDITION

---

9. PROCESSING RATE CONTROL (NEW)

---

OLD:

* Fixed implicit rate

NEW:

* Added:

  * min_processing_interval = 0.08

IMPACT:

* Controls FPS (~12.5 FPS)
* Prevents overload

VERDICT: IMPORTANT STABILITY FIX

---

10. CLOUD OPTIMIZATION FLAG (NEW)

---

OLD:

* Full data always sent

NEW:

* Added:

  * cloud_roi_headers_only = true

IMPACT:

* Reduces payload size
* Improves latency

VERDICT: EFFICIENCY BOOST

---

11. COMMENT REMOVAL / CLEANUP

---

OLD:

* Commented unused configs

NEW:

* Removed unnecessary comments

IMPACT:

* Cleaner config

- Slightly less documentation

VERDICT: NEUTRAL

---

12. PARAMETER CONSISTENCY

---

OLD:

* Mismatch with upgraded code

NEW:

* Matches:

  * tested_load.py
  * bandwidth node
  * new architecture

IMPACT:

* No runtime mismatch
* System coherence

VERDICT: VERY IMPORTANT



---

## PERFORMANCE IMPACT

Flexibility:         MUCH BETTER
Control:             MUCH BETTER
Reliability:         BETTER
Scalability:         MUCH BETTER

---

## CRITICAL FIXES

1. Bandwidth threshold bug fixed
2. Monitoring node properly configured
3. Load balancer parameters aligned

---



## SUMMARY

In [ ]:
1. **Launch File Start**

   * XML declaration:

     ```
     <?xml version="1.0"?>
     ```
   * `<launch>` → root tag for ROS launch system.

---

2. **Argument: only_line_follow**

   * Defined:

     ```
     <arg name="only_line_follow" default="false"/>
     ```
   * Controls:

     * whether full system runs OR only lane following.

---

3. **Include External Launch Files**

   * Starts core hardware/system components:

   a) Depth Camera:

   ```
   depth_cam.launch
   ```

   → initializes camera input

   b) Servo Controllers:

   ```
   start.launch
   ```

   → initializes servo motors

   c) Robot Controller:

   ```
   hiwonder_controller.launch
   ```

   → base robot control system

---

4. **YOLO Node Configuration**

   * Argument:

     ```
     yolo_node_name = yolov5_lb
     ```

   * Sets detection classes:

     * traffic-related classes
     * COCO object classes

---

5. **YOLO Node Execution**

   ```
   <node unless="$(arg only_line_follow)">
   ```

   * Runs only if NOT in line-follow mode.

   Parameters:

   * `use_depth_cam` → enable camera input
   * `engine` → TensorRT model for traffic signs
   * `od_engine` → object detection model
   * `lib` → custom TensorRT plugin
   * `conf_thresh` → detection threshold
   * `frame_skip` → skip frames for performance
   * `cache_ttl` → reuse detections for 0.25s
   * `image_path` → fallback/test image

---

6. **Self Driving Node**

   ```
   self_dri.py
   ```

   * Main driving logic
   * Parameter:

     * `only_line_follow` → limits behavior

---

7. **Bandwidth Monitor Node**

   ```
   bandwidth_monitor_node.py
   ```

   * Measures:

     * upload/download
     * RTT
     * jitter

   Parameters:

   * interface = wlan0
   * edge_host = 192.168.20.16
   * sample_period = 1 sec
   * rtt_window = 12 samples

---

8. **Lane Detection Node**

   ```
   navigation.py
   ```

   * Performs lane detection
   * Uses depth camera

---

9. **Load Balancer Node (MAIN LOGIC)**

   ```
   tested_load.py
   ```

   * Core decision system

---

10. **Applications Config**

```
['lane_detection','collision_avoidance','traffic_sign_detection']
```

* Defines which tasks are managed.

---

11. **Edge Server Config**

```
http://192.168.20.16:30052/processimage
```

* Target server for offloading processing.

---

12. **Mode Control**

```
only_line_follow
```

* Passed to load balancer.

---

13. **Resource Thresholds**

* CPU/GPU/RAM usage limit:

  ```
  resource_threshold = 80%
  ```

---

14. **Bandwidth Thresholds**

* High bandwidth:

  ```
  7 Mbps
  ```
* Low bandwidth:

  ```
  2 Mbps
  ```

---

15. **Resolution Settings**

* Edge:

  ```
  640 width
  ```
* Cloud:

  ```
  960 x 1080
  ```
* Low-res scaling:

  ```
  0.65
  ```

---

16. **Latency + Decision Thresholds**

* Edge timeout:

  ```
  2 sec
  ```
* Cloud delay threshold:

  ```
  0.75 sec
  ```
* Dual path threshold:

  ```
  0.60
  ```
* Dual margin:

  ```
  0.15
  ```

---

17. **Power Constraint**

```
power_budget_mw = 3200
```

* Used to limit onboard processing.

---

18. **Processing Control**

* Minimum frame interval:

  ```
  0.08 sec
  ```
* Limits FPS (~12.5 FPS max)

---

19. **Cloud Queue Control**

* Max pending requests:

  ```
  4
  ```

---

20. **ROI Optimization**

```
cloud_roi_headers_only = true
```

* Sends region info instead of full image when possible.

---

21. **Request Limit**

```
max_requests = 100
```

* Caps total cloud requests.

---

22. **Overall System Flow**

* Camera → YOLO + Lane Detection
* Bandwidth Monitor → Network stats
* Load Balancer:

  * collects:

    * system usage
    * network
    * scene
  * decides:
    → onboard OR edge OR both
* Edge server → returns results
* Fusion → final output

---

23. **Execution Order (simplified)**

* Start hardware (camera + servos)
* Start detection nodes
* Start bandwidth monitor
* Start load balancer
* System runs continuously

---

24. **Key Role of This File**

* This file:
  → **connects all nodes together**
  → defines full system architecture
* It is:
  👉 system orchestrator
